In [1]:
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings
import requests
from langchain_core.documents import Document

from langchain_community.document_loaders import (
    DirectoryLoader,
    Docx2txtLoader,
    PyPDFLoader,
    TextLoader,
)

C:\Users\alejg\AppData\Local\Temp\ipykernel_9664\2635050881.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


# config

In [2]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0) 

embeddings_model = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434",
)

vector_db = Chroma(
    collection_name="tourist_info",
    embedding_function=embeddings_model,
    persist_directory="./chroma_db"
)

In [3]:


HEADERS = {
    "User-Agent": "AlejandroRAGBot/1.0 (contacto: tu-email@ejemplo.com)"
}

def load_wikipedia_article(title: str, lang: str = "es", verbose: bool = False) -> Document:
    url = f"https://{lang}.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,   # texto plano, sin markup wiki
        "titles": title,
        "format": "json",
        "redirects": 1,     # sigue redirects (ej. si el título real es distinto)
    }
    resp = requests.get(url, params=params, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    if verbose:
        print(data)
    pages = data["query"]["pages"]
    page = next(iter(pages.values()))  # solo hay una página en el dict

    if "missing" in page:
        raise ValueError(f"No se encontró el artículo: {title}")

    return Document(
        page_content=page.get("extract", ""),
        metadata={
            "title": page.get("title"),
            "source": f"https://{lang}.wikipedia.org/wiki/{page.get('title', '').replace(' ', '_')}",
        },
    )


#load_wikipedia_article(title="Paestum", lang="es",verbose=True)


# Index data from wikipedia

In [4]:
doc = load_wikipedia_article("Paestum", lang="es")
print(len(doc.page_content))  # comprueba que trae el artículo completo

wikipedia_chunks = text_splitter.split_documents([doc])
vector_db.add_documents(wikipedia_chunks)

7079


['2f639442-849c-4832-9bb8-9fd58cb4b4c4',
 '318974e7-0e32-4473-abd3-cb3bc67c4597',
 '8aa85f8f-5dc2-4745-a1c2-9caed6e9a947',
 '9a023d73-6ed4-443e-a4be-5b1905e1167e',
 '47bc726b-0db1-426d-a7c8-ec9320606e73',
 'dd05a4d2-2b2f-45c0-937d-0c375beaac4f',
 '4ac608f9-39a0-4ae1-8559-794bff381a8f',
 'eeee410b-c1a6-464f-94e5-167955488d60',
 '9f4d17d4-c73b-4132-bc45-269c680b5837',
 '364093be-22d5-467e-9cb7-43fc5de847ee',
 'e212d46a-9546-4246-9723-70194517ccdf',
 'e3a4e68e-d2e3-4d57-8806-97b001d80648',
 '4759003d-7126-40e6-9a60-e80a2387501b',
 '4fa35871-7b7e-4ff3-abf5-5a90b0dcebcc',
 '1714284b-9a3d-4719-9a8e-5d0f9c770bb9',
 '877fbbe3-812a-4bfe-8276-9d43d5fc5064',
 '0feec233-0308-4edc-97e5-11f79241c6f3',
 'e06931e4-5dfb-4e6f-952c-b864f7108710',
 '69f51cd1-d1c3-48c7-9b4c-65637bce4108',
 '49e972df-8d40-42fd-80d4-d09cdf7847bc',
 '41a1e396-5e7c-4b7f-9a30-eb657af3802d',
 'ef320f39-1022-4667-92bf-578688b3ba2e',
 'dd4546ef-a3c2-4e57-a508-73a8e0cac26f']

# INDEX TXT PDF AND WORD

In [5]:
def split_and_import(loader):
    chunks = text_splitter.split_documents(loader.load())
    if not chunks:
        print(f"No chunks created by {loader}")
        return
    vector_db.add_documents(chunks)
    print(f"Ingested {len(chunks)} chunks created by {loader}")

In [6]:


word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)

Ingested 8 chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x000001F5FD8E7C80>
Ingested 29 chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x000001F5FC626870>
Ingested 1 chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x000001F5D9414E00>


# Index from a folder

In [7]:


folder_path = "CilentoTouristInfo"

loaders = [
    DirectoryLoader(
        folder_path,
        glob="**/*.docx",
        loader_cls=Docx2txtLoader,
    ),
    DirectoryLoader(
        folder_path,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
    ),
    DirectoryLoader(
        folder_path,
        glob="**/*.txt",
        loader_cls=TextLoader,
    ),
]

documents = []
for loader in loaders:
    documents.extend(loader.load())

chunks = text_splitter.split_documents(documents)

if chunks:
    vector_db.add_documents(chunks)
    print(f"Ingested {len(chunks)} chunks")
else:
    print("No se encontraron documentos o ninguno produjo texto.")

Ingested 212 chunks


In [8]:
query = "Where was Poseidonia and who renamed it to Paestum?"
results = vector_db.similarity_search(query, 4)   #1
print(results)

[Document(id='ef38a7cc-765c-4fe8-909a-b671a490fb31', metadata={'source': 'Paestum/Paestum-Britannica.docx'}, page_content='Poseidonia was probably founded about 600\xa0BC\xa0by Greek colonists from\xa0Sybaris, along the\xa0Gulf of Taranto, and it had become a flourishing town by 540, judging from its temples. After many years’ resistance the city came under the domination of the\xa0Lucanians\xa0(an\xa0indigenous\xa0Italic people) sometime before 400\xa0BC, after which its name was changed to Paestum. Alexander, the king of Epirus, defeated the Lucanians at Paestum about 332\xa0BC, but the city remained Lucanian until 273, when it came under'), Document(id='e85decc1-c0c4-464d-93dd-0db9b99523e7', metadata={'source': 'Paestum/Paestum-Britannica.docx'}, page_content='Paestum\n\nancient city, Italy\n\nPrint\xa0Cite\xa0Share\xa0Feedback\xa0\n\nAlso known as: Poseidonia\n\nWritten and fact-checked by\xa0\n\n\n\n\n\nThe Editors of Encyclopaedia Britannica\n\nLast Updated:\xa0Article History\n\